In [ ]:
STUDENT_ID = "230103041"

assert STUDENT_ID != "YOUR_STUDENT_ID_HERE", "Set STUDENT_ID first!"

import os
REPO = f"cuda-lab-02-{STUDENT_ID}"
os.makedirs(f"/content/{REPO}", exist_ok=True)
os.chdir(f"/content/{REPO}")
print("Working folder:", os.getcwd())

Working folder: /content/cuda-lab-02-230103041


In [ ]:
!nvidia-smi

Fri Oct  9 09:09:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from numba import cuda
d = cuda.get_current_device()
gpu_name = d.name.decode() if isinstance(d.name, bytes) else d.name
print("GPU:", gpu_name, "| Compute capability:", d.compute_capability)

GPU: Tesla T4 | Compute capability: ComputeCapability(major=7, minor=5)


## Step 1 — Create the repository files

In [ ]:
%%writefile task1_divergence.py

import time
import numpy as np
from numba import cuda

N = 2 ** 20
ITERS = 1000
THREADS_PER_BLOCK = 256

MUL = np.float32(1.0001)
ADD = np.float32(0.0001)


@cuda.jit
def kernel_a_uniform(y):
    idx = cuda.grid(1)
    if idx < y.shape[0]:
        v = y[idx]
        for _ in range(ITERS):
            v = v * MUL + ADD
        y[idx] = v


@cuda.jit
def kernel_b_divergent(y):
    idx = cuda.grid(1)
    if idx < y.shape[0]:
        v = y[idx]
        if idx % 2 == 0:
            for _ in range(ITERS):
                v = v * MUL + ADD
        else:
            for _ in range(ITERS):
                v = (v - ADD) / MUL
        y[idx] = v


@cuda.jit
def kernel_c_warp_aligned(y):
    idx = cuda.grid(1)
    if idx < y.shape[0]:
        v = y[idx]
        warp_id = idx // 32
        if warp_id % 2 == 0:
            for _ in range(ITERS):
                v = v * MUL + ADD
        else:
            for _ in range(ITERS):
                v = (v - ADD) / MUL
        y[idx] = v


def bench(kernel, d_y, blocks, trials=10):
    kernel[blocks, THREADS_PER_BLOCK](d_y)
    cuda.synchronize()
    times = []
    for _ in range(trials):
        cuda.synchronize()
        t0 = time.perf_counter()
        kernel[blocks, THREADS_PER_BLOCK](d_y)
        cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000.0)
    return float(np.mean(times))


def main():
    blocks = (N + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK
    h_y = np.ones(N, dtype=np.float32)
    d_y = cuda.to_device(h_y)

    t_a = bench(kernel_a_uniform, d_y, blocks)
    t_b = bench(kernel_b_divergent, d_y, blocks)
    t_c = bench(kernel_c_warp_aligned, d_y, blocks)

    print(f"GPU: {cuda.get_current_device().name}")
    print(f"N = {N:,}, iterations per element = {ITERS}, avg of 10 trials\n")
    print("| Kernel | Description | Avg Kernel Time (ms) | Ratio vs A |")
    print("|---|---|---|---|")
    print(f"| A | Uniform path | {t_a:.4f} | 1.00x |")
    print(f"| B | Full divergence (idx % 2) | {t_b:.4f} | {t_b / t_a:.2f}x |")
    print(f"| C | Warp-aligned (idx // 32) | {t_c:.4f} | {t_c / t_a:.2f}x |")
    print(f"\nDivergence penalty B/C = {t_b / t_c:.2f}x")


if __name__ == "__main__":
    main()

Writing task1_divergence.py


In [ ]:
%%writefile task2_stencil_1d.py
import numpy as np
from numba import cuda

THREADS_PER_BLOCK = 256


@cuda.jit
def stencil_1d(d_in, d_out, N):
    idx = cuda.grid(1)
    if idx < N:

        if idx == 0:
            left = d_in[0]
        else:
            left = d_in[idx - 1]
        if idx == N - 1:
            right = d_in[N - 1]
        else:
            right = d_in[idx + 1]
        d_out[idx] = 0.25 * left + 0.5 * d_in[idx] + 0.25 * right


def run_stencil(h_in):
    h_in = np.ascontiguousarray(h_in, dtype=np.float32)
    N = h_in.shape[0]
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array(N, dtype=np.float32)
    blocks = (N + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK
    stencil_1d[blocks, THREADS_PER_BLOCK](d_in, d_out, N)
    cuda.synchronize()
    return d_out.copy_to_host()


def cpu_stencil(arr):
    padded = np.pad(arr, (1, 1), mode='edge')
    return 0.25 * padded[:-2] + 0.5 * padded[1:-1] + 0.25 * padded[2:]


if __name__ == "__main__":
    N = 100_007
    np.random.seed(0)
    h_in = np.random.rand(N).astype(np.float32)
    h_out_gpu = run_stencil(h_in)
    cpu_ref = cpu_stencil(h_in)
    assert np.allclose(h_out_gpu, cpu_ref, atol=1e-4),
    max_delta = float(np.max(np.abs(h_out_gpu - cpu_ref)))
    print(f"TASK 2 PASSED: MAX DELTA = {max_delta}")

Writing task2_stencil_1d.py


In [ ]:
%%writefile task3_grid_stride.py
import numpy as np
from numba import cuda

THREADS_PER_BLOCK = 256
BLOCKS_PER_GRID = 64


@cuda.jit
def grid_stride_scale_kernel(d_arr, factor, N):
    start = cuda.grid(1)
    stride = cuda.gridsize(1)
    for i in range(start, N, stride):
        d_arr[i] = d_arr[i] * factor


def run_grid_stride(h_arr, factor):
    h_arr = np.ascontiguousarray(h_arr, dtype=np.float32)
    N = h_arr.shape[0]
    d_arr = cuda.to_device(h_arr)
    grid_stride_scale_kernel[BLOCKS_PER_GRID, THREADS_PER_BLOCK](d_arr, np.float32(factor), N)
    cuda.synchronize()
    return d_arr.copy_to_host()


if __name__ == "__main__":
    N = 2 ** 24
    factor = 4.25
    h_arr = np.ones(N, dtype=np.float32)
    result = run_grid_stride(h_arr, factor)
    assert result.shape[0] == N
    assert np.all(result == np.float32(factor)), "Task 3: not all elements match factor"
    print(f"TASK 3 PASSED: all {N:,} elements == {factor} "
          f"(threads launched: {BLOCKS_PER_GRID * THREADS_PER_BLOCK:,})")

Writing task3_grid_stride.py


In [ ]:
%%writefile task4_sobel_2d.py
import numpy as np
from numba import cuda

THREADS_2D = (16, 16)


@cuda.jit
def sobel_x_kernel(d_in, d_out, rows, cols):
    col, row = cuda.grid(2)
    if row < rows and col < cols:
        if 0 < row < rows - 1 and 0 < col < cols - 1:
            d_out[row, col] = (
                -1.0 * d_in[row - 1, col - 1] + 1.0 * d_in[row - 1, col + 1]
                - 2.0 * d_in[row, col - 1] + 2.0 * d_in[row, col + 1]
                - 1.0 * d_in[row + 1, col - 1] + 1.0 * d_in[row + 1, col + 1]
            )
        else:
            d_out[row, col] = 0.0


def run_sobel(h_img):
    h_img = np.ascontiguousarray(h_img, dtype=np.float32)
    rows, cols = h_img.shape
    d_in = cuda.to_device(h_img)
    d_out = cuda.device_array((rows, cols), dtype=np.float32)
    blocks_2d = ((cols + THREADS_2D[0] - 1) // THREADS_2D[0],
                 (rows + THREADS_2D[1] - 1) // THREADS_2D[1])
    sobel_x_kernel[blocks_2d, THREADS_2D](d_in, d_out, rows, cols)
    cuda.synchronize()
    return d_out.copy_to_host()


def cpu_sobel(img):
    out = np.zeros_like(img)
    a = img
    out[1:-1, 1:-1] = (-a[:-2, :-2] + a[:-2, 2:]
                       - 2 * a[1:-1, :-2] + 2 * a[1:-1, 2:]
                       - a[2:, :-2] + a[2:, 2:])
    return out


if __name__ == "__main__":
    np.random.seed(0)
    img = np.random.rand(2048, 2048).astype(np.float32)   # 2048 x 2048
    res = run_sobel(img)
    ref = cpu_sobel(img)
    assert np.allclose(res, ref, atol=1e-4), "Task 4 mismatch vs CPU reference"
    assert np.all(res[0, :] == 0) and np.all(res[-1, :] == 0)
    assert np.all(res[:, 0] == 0) and np.all(res[:, -1] == 0)
    print(f"TASK 4 PASSED: shape {res.shape}, grid = 128 x 128 blocks, "
          f"max delta = {float(np.max(np.abs(res - ref)))}")

Writing task4_sobel_2d.py


In [ ]:
%%writefile verify_submission.py

import sys
import hashlib
import numpy as np
from numba import cuda


def main():
    print("=" * 60)
    print("RUNNING CUDA LAB 02 AUTONOMOUS VERIFICATION")
    print("=" * 60)

    student_id = input("Enter your Student ID: ").strip()
    if not student_id:
        print("[FAIL] Student ID cannot be empty.")
        sys.exit(1)

    # 1. Verify Task 2
    try:
        import task2_stencil_1d as t2
        N = 10007
        test_in = np.sin(np.linspace(0, 10, N)).astype(np.float32)
        h_gpu = t2.run_stencil(test_in)
        h_cpu = t2.cpu_stencil(test_in)
        assert np.allclose(h_gpu, h_cpu, atol=1e-4), "Task 2 output mismatch against CPU"
        print("[PASS] Task 2 (1D Stencil & Clamping)")
    except Exception as e:
        print(f"[FAIL] Task 2: {e}")
        sys.exit(1)

    # 2. Verify Task 3
    try:
        import task3_grid_stride as t3
        N = 100000
        test_arr = np.ones(N, dtype=np.float32)
        factor = 4.25
        res = t3.run_grid_stride(test_arr, factor)
        assert np.allclose(res, factor), "Task 3 elements not uniformly scaled"
        print("[PASS] Task 3 (Grid-Stride Scaling)")
    except Exception as e:
        print(f"[FAIL] Task 3: {e}")
        sys.exit(1)

    # 3. Verify Task 4
    try:
        import task4_sobel_2d as t4
        test_img = np.ones((64, 64), dtype=np.float32)
        sobel_res = t4.run_sobel(test_img)
        # Uniform image must have zero interior gradient
        assert np.max(np.abs(sobel_res[1:-1, 1:-1])) < 1e-5, "Task 4 interior gradient != 0 for flat field"
        assert np.all(sobel_res[0, :] == 0.0), "Task 4 border rows not zeroed"
        assert np.all(sobel_res[:, 0] == 0.0), "Task 4 border columns not zeroed"
        print("[PASS] Task 4 (2D Sobel Horizontal)")
    except Exception as e:
        print(f"[FAIL] Task 4: {e}")
        sys.exit(1)

    # Cryptographic Checksum Token Generation
    hasher = hashlib.sha256()
    hasher.update(student_id.encode('utf-8'))
    try:
        device_name = cuda.get_current_device().name
        hasher.update(device_name if isinstance(device_name, bytes) else device_name.encode('utf-8'))
    except Exception:
        hasher.update(b"UNKNOWN_CUDA_DEVICE")
    hasher.update(h_gpu[:32].tobytes())
    hasher.update(sobel_res[:8, :8].tobytes())
    token = hasher.hexdigest()[:20].upper()

    print("\n" + "=" * 60)
    print("VERIFICATION SUCCESSFUL")
    print(f"OFFICIAL SUBMISSION TOKEN: {token}")
    print("=" * 60)
    print("Copy this token directly into your README.md.\n")


if __name__ == "__main__":
    main()

Writing verify_submission.py


## Step 2 — Run all tasks on the T4

In [ ]:
import subprocess, sys, re

def run(script, stdin_text=None):
    r = subprocess.run([sys.executable, script], input=stdin_text, capture_output=True, text=True)
    print(f"$ python {script}   (exit code {r.returncode})")
    print(r.stdout)
    if r.returncode != 0:
        print("[stderr]\n" + r.stderr[-2000:])
        raise RuntimeError(f"{script} failed - read the error above")
    return r.stdout

outs = {}
for s in ["task1_divergence.py", "task2_stencil_1d.py", "task3_grid_stride.py", "task4_sobel_2d.py"]:
    outs[s] = run(s)

$ python task1_divergence.py   (exit code 0)
GPU: Tesla T4
N = 1,048,576, iterations per element = 1000, avg of 10 trials

| Kernel | Description | Avg Kernel Time (ms) | Ratio vs A |
|---|---|---|---|
| A | Uniform path | 2.9660 | 1.00x |
| B | Full divergence (idx % 2) | 9.4525 | 3.19x |
| C | Warp-aligned (idx // 32) | 3.6944 | 1.25x |

Divergence penalty B/C = 2.56x

$ python task2_stencil_1d.py   (exit code 0)
TASK 2 PASSED: MAX DELTA = 5.960464477539063e-08

$ python task3_grid_stride.py   (exit code 0)
TASK 3 PASSED: all 16,777,216 elements == 4.25 (threads launched: 16,384)

$ python task4_sobel_2d.py   (exit code 0)
TASK 4 PASSED: shape (2048, 2048), grid = 128 x 128 blocks, max delta = 4.76837158203125e-07



## Step 3 — Official verification and token

In [ ]:
out_v = run("verify_submission.py", STUDENT_ID + "\n")
m = re.search(r"TOKEN: ([0-9A-F]{20})", out_v)
assert m, "Token not found - verification did not succeed"
token = m.group(1)
print("YOUR TOKEN:", token)

$ python verify_submission.py   (exit code 0)
RUNNING CUDA LAB 02 AUTONOMOUS VERIFICATION
Enter your Student ID: [PASS] Task 2 (1D Stencil & Clamping)
[PASS] Task 3 (Grid-Stride Scaling)
[PASS] Task 4 (2D Sobel Horizontal)

VERIFICATION SUCCESSFUL
OFFICIAL SUBMISSION TOKEN: 8A49EF0A3557D2B495AA
Copy this token directly into your README.md.


YOUR TOKEN: 8A49EF0A3557D2B495AA


## Step 5 — Check the folder and download

In [ ]:
print(sorted(os.listdir(".")))
import shutil
shutil.rmtree("__pycache__", ignore_errors=True)
os.chdir("/content")
shutil.make_archive(REPO, "zip", root_dir="/content", base_dir=REPO)
from google.colab import files
files.download(f"/content/{REPO}.zip")

['README.md', '__pycache__', 'task1_divergence.py', 'task2_stencil_1d.py', 'task3_grid_stride.py', 'task4_sobel_2d.py', 'verify_submission.py']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>